# __Data Cleaning__

## Import Data

In [1]:
from data_loader import load_data
import pandas as pd 

# Data
data = load_data(filename='data_inspection.parquet', 
                 revision='data-inspection')
display(data.info(),data)

<class 'pandas.DataFrame'>
RangeIndex: 27509 entries, 0 to 27508
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_persona    27509 non-null  string
 1   id_hogar      27509 non-null  string
 2   id_vivienda   27509 non-null  string
 3   ciudad        27509 non-null  int64 
 4   area          27509 non-null  int64 
 5   conglomerado  27509 non-null  int64 
 6   panelm        27509 non-null  int64 
 7   vivienda      27509 non-null  int64 
 8   hogar         27509 non-null  int64 
 9   p01           27509 non-null  int64 
 10  periodo       27509 non-null  int64 
 11  fexp          27509 non-null  str   
 12  estrato       27509 non-null  int64 
 13  upm           27509 non-null  int64 
 14  pt3b          27509 non-null  str   
 15  ingrl         27509 non-null  str   
dtypes: int64(10), str(3), string(3)
memory usage: 5.3 MB


None

,id_persona,id_hogar,id_vivienda,ciudad,area,conglomerado,panelm,vivienda,hogar,p01,periodo,fexp,estrato,upm,pt3b,ingrl
0,1015000100206801101,10150001002068011,1015000100206801,10150,1,1002,68,1,1,1,202507,"402,91638244211",2713,10150001002,1,672
1,1015000100206801102,10150001002068011,1015000100206801,10150,1,1002,68,1,1,2,202507,"402,91638244211",2713,10150001002,0,120
2,1015000100206801103,10150001002068011,1015000100206801,10150,1,1002,68,1,1,3,202507,"402,91638244211",2713,10150001002,2,
3,1015000100206801104,10150001002068011,1015000100206801,10150,1,1002,68,1,1,4,202507,"402,91638244211",2713,10150001002,1,
4,1015000100206803101,10150001002068031,1015000100206803,10150,1,1002,68,3,1,1,202507,"402,91638244211",2713,10150001002,6,560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27504,24035200270305407101,240352002703054071,24035200270305407,240352,1,2703,54,7,1,1,202507,"649,399292595133",2412,240352002703,4,500
27505,24035200270305407102,240352002703054071,24035200270305407,240352,1,2703,54,7,1,2,202507,"649,399292595133",2412,240352002703,2,182
27506,24035200270305407103,240352002703054071,24035200270305407,240352,1,2703,54,7,1,3,202507,"649,399292595133",2412,240352002703,2,150
27507,24035200270305407104,240352002703054071,24035200270305407,240352,1,2703,54,7,1,4,202507,"649,399292595133",2412,240352002703,3,470


## Cleaning Data

### __1. ID Variables__

In [2]:
# Filter

id_columns = [
    'id_persona', 'id_hogar', 'id_vivienda', 'ciudad', 'area',
    'conglomerado', 'panelm', 'vivienda', 'hogar', 'p01', 'periodo'
]

# Treatment with NA
for col in id_columns:
    data[col] = data[col].astype('string').str.strip().replace('', pd.NA)

In [3]:
# Missing identifiers
display(data[id_columns].isna().sum())

# Repeated person-period identifiers
duplicates = data.duplicated(['id_persona', 'periodo'], keep=False)
display(data.loc[duplicates].sort_values(['id_persona', 'periodo']))

id_persona      0
id_hogar        0
id_vivienda     0
ciudad          0
area            0
conglomerado    0
panelm          0
vivienda        0
hogar           0
p01             0
periodo         0
dtype: int64

,id_persona,id_hogar,id_vivienda,ciudad,area,conglomerado,panelm,vivienda,hogar,p01,periodo,fexp,estrato,upm,pt3b,ingrl


## __2. Sampling Variables__

In [4]:
# Clean sampling identifiers
for col in ['estrato', 'upm']:
    data[col] = data[col].astype('string').str.strip().replace('', pd.NA)

# Convert sampling weights
weights = data['fexp'].astype('string').str.strip().replace('', pd.NA)
data['fexp'] = pd.to_numeric(
    weights.str.replace(',', '.', regex=False), errors='raise'
)

In [5]:
# Validate survey design
display(data[['fexp', 'estrato', 'upm']].isna().sum())
display(data.loc[data['fexp'].le(0), ['id_persona', 'fexp']])

fexp       0
estrato    0
upm        0
dtype: int64

,id_persona,fexp


## __3. Variables of interest__
  
- $X: $ Internet Hours  
- $Y: $ Labor Income 

In [6]:
data[['pt3b','ingrl']].info()

<class 'pandas.DataFrame'>
RangeIndex: 27509 entries, 0 to 27508
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   pt3b    27509 non-null  str  
 1   ingrl   27509 non-null  str  
dtypes: str(2)
memory usage: 510.0 KB


#### 3.1 __Internet Hours__

In [7]:
# Unique Values
data['pt3b'].value_counts(dropna=False)

pt3b
2      5577
       5082
1      4341
3      4033
4      2797
5      1764
888    1255
6      1088
8       571
0       398
7       248
10      187
9        65
12       64
15       13
11       10
14        7
13        6
16        2
18        1
Name: count, dtype: int64

##### __3.1.1 Internet Use__

In [8]:
# Import variable 
# pt2a : Internet use

data_2 = load_data(filename='enemdu_persona_tics_2025_07.csv', 
                           revision='main')
display(data_2['pt2a'].info(),data_2['pt2a'])

<class 'pandas.Series'>
RangeIndex: 27509 entries, 0 to 27508
Series name: pt2a
Non-Null Count  Dtype
--------------  -----
27509 non-null  str  
dtypes: str(1)
memory usage: 243.1 KB


None

0        1
1        1
2        1
3        1
4        1
        ..
27504    1
27505    1
27506    1
27507    1
27508    1
Name: pt2a, Length: 27509, dtype: str

In [9]:
# Unique values
data_2['pt2a'].value_counts(dropna= False)

pt2a
1     21172
2      3279
       1803
88     1255
Name: count, dtype: int64

In [10]:
# Compare internet-use responses with reported hours
display(pd.crosstab(
    data_2['pt2a'].str.strip().replace('', 'Blank'),
    data_2['pt3b'].str.strip().replace('', 'Blank')
))

pt3b,0,1,10,11,12,13,14,15,16,18,2,3,4,5,6,7,8,888,9,Blank
pt2a,,,,,,,,,,,,,,,,,,,,
1,398,4341,187,10,64,6,7,13,2,1,5577,4033,2797,1764,1088,248,571,0,65,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3279
88,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1255,0,0
Blank,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1803


###### __Conclusions__
| Condition | Interpretation |
|---|---|
| `pt2a = 88` and `pt3b = 888` | Nonresponse: the respondent was unsuitable or could not be contacted. |
| `pt2a = 2` and blank `pt3b` | Structural missingness due to questionnaire routing: hours were not asked because the person did not use the internet. |
| Blank `pt2a` and blank `pt3b` | Potentially outside the ICT module's target population; verify whether these individuals are under five years old. |


In [11]:
import numpy as np

responses = data_2[['id_persona', 'pt2a', 'pt3b']].astype('string')
responses = responses.apply(lambda col: col.str.strip()).replace('', pd.NA)

conditions = [
    responses['pt2a'].eq('88') & responses['pt3b'].eq('888'),
    responses['pt2a'].eq('2') & responses['pt3b'].isna(),
    responses['pt2a'].isna() & responses['pt3b'].isna(),
    responses['pt2a'].eq('1') & responses['pt3b'].notna()
]

responses['internet_hours_status'] = np.select(
    [condition.fillna(False).to_numpy(dtype=bool) for condition in conditions],
    ['Nonresponse', 'Not applicable', 'Potentially out of scope', 'Observed'],
    default='Review'
)

status = responses.set_index('id_persona')['internet_hours_status']
data['internet_hours_status'] = data['id_persona'].astype('string').map(status)

data['internet_hours_status'].value_counts(dropna=False)

internet_hours_status
Observed                    21172
Not applicable               3279
Potentially out of scope     1803
Nonresponse                  1255
Name: count, dtype: int64

In [12]:
display(data.info(),data['internet_hours_status'].head())

<class 'pandas.DataFrame'>
RangeIndex: 27509 entries, 0 to 27508
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id_persona             27509 non-null  string 
 1   id_hogar               27509 non-null  string 
 2   id_vivienda            27509 non-null  string 
 3   ciudad                 27509 non-null  string 
 4   area                   27509 non-null  string 
 5   conglomerado           27509 non-null  string 
 6   panelm                 27509 non-null  string 
 7   vivienda               27509 non-null  string 
 8   hogar                  27509 non-null  string 
 9   p01                    27509 non-null  string 
 10  periodo                27509 non-null  string 
 11  fexp                   27509 non-null  Float64
 12  estrato                27509 non-null  string 
 13  upm                    27509 non-null  string 
 14  pt3b                   27509 non-null  str    
 15  ingrl        

None

0    Observed
1    Observed
2    Observed
3    Observed
4    Observed
Name: internet_hours_status, dtype: str

In [13]:
# Convert reported hours to nullable integers
hours = data['pt3b'].astype('string').str.strip().replace(['', '888'], pd.NA)
data['pt3b'] = pd.to_numeric(hours, errors='raise').astype('Int64')

data['pt3b'].value_counts()

pt3b
2     5577
1     4341
3     4033
4     2797
5     1764
6     1088
8      571
0      398
7      248
10     187
9       65
12      64
15      13
11      10
14       7
13       6
16       2
18       1
Name: count, dtype: Int64

In [14]:
# Validate reported hours by response status
display(data.groupby('internet_hours_status')['pt3b'].agg(['size', 'count', 'min', 'max']))

,size,count,min,max
internet_hours_status,,,,
Nonresponse,1255,0,<NA>,<NA>
Not applicable,3279,0,<NA>,<NA>
Observed,21172,21172,0,18
Potentially out of scope,1803,0,<NA>,<NA>


#### __3.2 Labor Income__

In [15]:
# Inspect labor-income values
data['ingrl'].value_counts(dropna=False)

ingrl
        15435
470       536
400       402
500       396
300       378
        ...  
227         1
86          1
2020        1
4010        1
182         1
Name: count, Length: 1325, dtype: int64

In [16]:
# Count blanks, zeros, and potential special codes
income = data['ingrl'].astype('string').str.strip()

display(income.value_counts(dropna=False).reindex(
    ['', '-1', '0', '999999'], fill_value=0
).rename(index={'': 'Blank'}))

ingrl
Blank     15435
-1          129
0           157
999999       39
Name: count, dtype: int64[pyarrow]

In [17]:
# Compare employment status with blank labor-income responses
display(pd.crosstab(
    data_2['empleo'].astype('string').str.strip().replace('', 'Blank'),
    data_2['ingrl'].astype('string').str.strip().fillna('').eq(''),
    rownames=['employment_code'],
    colnames=['blank_income']
))

blank_income,False,True
employment_code,,
1,12066,1233
Blank,8,14202


In [18]:
# Inspect income records without an employment classification
employment = data_2['empleo'].astype('string').str.strip().fillna('')
income = data_2['ingrl'].astype('string').str.strip().fillna('')

display(data_2.loc[
    employment.eq('') & income.ne(''),
    ['id_persona', 'p03', 'condact', 'empleo', 'ingrl']
])

,id_persona,p03,condact,empleo,ingrl
3624,4015300010106501102,13,0,,95
4226,5015290040106808103,14,0,,0
5008,6105000140705406103,12,0,,0
17096,15035290060206507105,13,0,,128
17097,15035290060206507106,10,0,,128
25221,21075090050206809103,14,0,,80
25262,22015000070506806104,14,0,,45
25487,22015090041106802106,13,0,,26


In [19]:
# Inspect occupational categories among employed people with blank income
display(data_2.loc[
    employment.eq('1') & income.eq(''), 'p42'
].value_counts(dropna=False))

p42
7    1210
8      12
9      11
Name: count, dtype: int64

In [20]:
# Inspect activity status and ages among the remaining blank-income records
pending = data_2.loc[
    employment.eq('') & income.eq(''), ['condact', 'p03']
]

display(pending.groupby('condact', dropna=False)['p03'].agg(
    ['size', 'min', 'max']
))

,size,min,max
condact,,,
0,5607,0,14
7,517,16,69
8,62,17,76
9,8016,15,98


In [21]:
# Classify income responses using the original person records
income_responses = data_2[['id_persona', 'ingrl', 'condact', 'empleo', 'p42']].astype('string')
income_responses = income_responses.apply(lambda col: col.str.strip()).fillna('')
recorded = income_responses['ingrl'].ne('')

income_status = income_responses['condact'].map({
    '0': 'Blank: under 15',
    '7': 'Blank: open unemployment',
    '8': 'Blank: hidden unemployment',
    '9': 'Blank: economically inactive'
}).fillna('Review')
income_status.loc[recorded] = 'Observed'
income_status.loc[income_responses['ingrl'].eq('-1')] = 'Expenses exceed earnings'
income_status.loc[income_responses['ingrl'].eq('999999')] = 'Nonresponse'
income_status.loc[~recorded & income_responses['empleo'].eq('1')
                  & income_responses['p42'].isin(['7', '8', '9'])] = 'Blank: unpaid primary job'

income_responses['labor_income_status'] = income_status
income_lookup = income_responses.set_index('id_persona')
assert income_lookup.index.is_unique, 'Duplicate person IDs'
data['labor_income_status'] = (
    data['id_persona'].astype('string').map(income_lookup['labor_income_status'])
)
assert data['labor_income_status'].notna().all(), 'Some person IDs were not matched'

In [22]:
# Preserve original income responses, including spaces, before conversion
original_income = data_2.set_index(data_2['id_persona'].astype('string'))['ingrl']
data['ingrl_original'] = data['id_persona'].astype('string').map(original_income)

# Convert monetary amounts; retain observed zeros
income_values = data['ingrl_original'].astype('string').str.strip().replace(['', '-1', '999999'], pd.NA)
data['ingrl'] = pd.to_numeric(income_values, errors='raise').astype('Float64')

In [23]:
# Validate numeric income by response status
display(data.groupby('labor_income_status')['ingrl'].agg(['size', 'count', 'min', 'max']))

,size,count,min,max
labor_income_status,,,,
Blank: economically inactive,8016,0,<NA>,<NA>
Blank: hidden unemployment,62,0,<NA>,<NA>
Blank: open unemployment,517,0,<NA>,<NA>
Blank: under 15,5607,0,<NA>,<NA>
Blank: unpaid primary job,1233,0,<NA>,<NA>
Expenses exceed earnings,129,0,<NA>,<NA>
Nonresponse,39,0,<NA>,<NA>
Observed,11906,11906,0.0,17000.0


###### Conclusions

| Condition | Interpretation |
|---|---|
| Blank income, `condact=0` | Under age 15; 5,607 cases. |
| Blank income, `condact=7/8` | Open or hidden unemployment; 579 cases. |
| Blank income, `condact=9` | Economically inactive; 8,016 cases. |
| Blank income, `empleo=1`, `p42=7/8/9` | Unpaid primary job; 1,233 cases. |
| `ingrl=-1` | Expenses exceed earnings; 129 cases. |
| `ingrl=999999` | Nonresponse; 39 cases. |
| Recorded monetary values | 11,906 observations, including 157 zeros. |

Special codes and blanks were converted to `NA`, preserving their classification and original responses. Numeric income ranges from 0 to 17,000.

## Export Cleaned Data

In [25]:
# Store identifiers as strings to preserve every digit

data_export = data.astype({
    'id_persona': 'string',
    'id_hogar': 'string',
    'id_vivienda': 'string'
})
data_export

,id_persona,id_hogar,id_vivienda,ciudad,area,conglomerado,panelm,vivienda,hogar,p01,periodo,fexp,estrato,upm,pt3b,ingrl,internet_hours_status,labor_income_status,ingrl_original
0,1015000100206801101,10150001002068011,1015000100206801,10150,1,1002,68,1,1,1,202507,402.916382,2713,10150001002,1,672.0,Observed,Observed,672
1,1015000100206801102,10150001002068011,1015000100206801,10150,1,1002,68,1,1,2,202507,402.916382,2713,10150001002,0,120.0,Observed,Observed,120
2,1015000100206801103,10150001002068011,1015000100206801,10150,1,1002,68,1,1,3,202507,402.916382,2713,10150001002,2,<NA>,Observed,Blank: economically inactive,
3,1015000100206801104,10150001002068011,1015000100206801,10150,1,1002,68,1,1,4,202507,402.916382,2713,10150001002,1,<NA>,Observed,Blank: under 15,
4,1015000100206803101,10150001002068031,1015000100206803,10150,1,1002,68,3,1,1,202507,402.916382,2713,10150001002,6,560.0,Observed,Observed,560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27504,24035200270305407101,240352002703054071,24035200270305407,240352,1,2703,54,7,1,1,202507,649.399293,2412,240352002703,4,500.0,Observed,Observed,500
27505,24035200270305407102,240352002703054071,24035200270305407,240352,1,2703,54,7,1,2,202507,649.399293,2412,240352002703,2,182.0,Observed,Observed,182
27506,24035200270305407103,240352002703054071,24035200270305407,240352,1,2703,54,7,1,3,202507,649.399293,2412,240352002703,2,150.0,Observed,Observed,150
27507,24035200270305407104,240352002703054071,24035200270305407,240352,1,2703,54,7,1,4,202507,649.399293,2412,240352002703,3,470.0,Observed,Observed,470


In [ ]:
from pathlib import Path

# Save the cleaned dataset locally
file_path = Path('../data/processed/data_clean.parquet')
data_export.to_parquet(file_path, index=False)